<h1>Model pipeline to predict crop yeild</h1>

In [13]:
!conda env create -f environment.yml

19282.11s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda


Installing pip dependencies: | Ran pip subprocess with arguments:
['/home/sagemaker-user/.conda/envs/geospatial-python-crash-course/bin/python', '-m', 'pip', 'install', '-U', '-r', '/home/sagemaker-user/condaenv.l2eqj441.requirements.txt', '--exists-action=b']
Pip subprocess output:

done
#
# To activate this environment, use
#
#     $ conda activate geospatial-python-crash-course
#
# To deactivate an active environment, use
#
#     $ conda deactivate



In [17]:
!conda init

19426.29s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


no change     /opt/conda/condabin/conda
no change     /opt/conda/bin/conda
no change     /opt/conda/bin/activate
no change     /opt/conda/bin/deactivate
no change     /opt/conda/etc/profile.d/conda.sh
no change     /opt/conda/etc/fish/conf.d/conda.fish
no change     /opt/conda/shell/condabin/Conda.psm1
no change     /opt/conda/shell/condabin/conda-hook.ps1
no change     /opt/conda/lib/python3.12/site-packages/xontrib/conda.xsh
no change     /opt/conda/etc/profile.d/conda.csh
no change     /home/sagemaker-user/.bashrc
No action taken.


In [18]:
!conda activate geospatial-python-crash-course

19439.62s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



CondaError: Run 'conda init' before 'conda activate'



In [22]:
from pystac_client import Client # To query a STAC API endpoint
import planetary_computer # The Microsoft Data Catalog access library
import rasterio # Working with raster data
import matplotlib.pyplot as plt # For basic plotting
from matplotlib.colors import ListedColormap # To create a discrete color palette
import numpy as np # For working with numerical data
from terratorch.registry import BACKBONE_REGISTRY
import torch
import requests # Allows retriving data from the internet
import pandas as pd # For working with tabular data
import geopandas as gpd # For working with geospatial tabular data
import folium # For making interactive maps
from pystac_client import Client # To query STAC API endpoint
import planetary_computer # Microsoft Data Catalog library
from shapely.geometry import shape,GeometryCollection,box # To work with drawn geometry
from shapely.geometry.polygon import Polygon
import leafmap # A feature-rich interactive map allow us to save drawn geometry (among other things)
import geojson # To parse geospatial data
import json # to load and handle json structure
from shapely.ops import transform # The shapely transform module
import pyproj # A reprojection library
import rioxarray # To open raster images
from rioxarray import merge # To merge raster images
import localtileserver # To allow us to see a Geotiff in the web browser
import pytorch

ModuleNotFoundError: No module named 'pystac_client'

In [ ]:
Colorado_NAIP_data = "https://services.arcgis.com/LLVEmB8Lsae3Um4s/arcgis/rest/services/2023_CO_NAIP_Image_Dates_/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
Iowa_NAIP_data = "https://services.arcgis.com/LLVEmB8Lsae3Um4s/arcgis/rest/services/2025_Iowa_Image_Dates/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
Wisconsin_NAIP_data = "https://services.arcgis.com/LLVEmB8Lsae3Um4s/arcgis/rest/services/2024_Wisconsin_Image_Dates/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
Missouri_NAIP_data = "https://services.arcgis.com/LLVEmB8Lsae3Um4 s/arcgis/rest/services/2024_Missouri_Image_Dates/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
Nebraska_NAIP_data = "https://services.arcgis.com/LLVEmB8Lsae3Um4s/arcgis/rest/services/2024_Nebraska_Image_Dates/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"

In [21]:
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_600_tl", pretrained=True)

NameError: name 'BACKBONE_REGISTRY' is not defined

In [20]:
class createData():
    def __init__(self):
        train_transforms = [
            terratorch.datasets.transforms.FlattenTemporalIntoChannels(),
            albumentations.Flip(),
            albumentations.pytorch.transforms.ToTensorV2(),
            terratorch.datasets.transforms.UnflattenTemporalFromChannels(n_timesteps=NUM_FRAMES)]
        sagemaker_session = sagemaker.Session(default_bucket="sagemaker-us-west-2-851725341796")
        data_bucket = sagemaker_session.default_bucket()
        self.data_module = MultiTemporalCropClassificationDataModule(
            data_root=databucket,
            train_transform=train_transforms,
            expand_temporal_dimension=True,
            )
        self.data_module.setup("fit")
        sel.train_dataset = data_module.train_dataset
        



In [ ]:
data = createData()
data_module = MultiTemporalCropClassificationDataModule(
    batch_size=BATCH_SIZE,
    data_root=data.train_dataset,
    train_transform=train_transforms,
    reduce_zero_label=True,
    expand_temporal_dimension=True,
    num_workers=7,
    use_metadata=True
    )

In [ ]:
class buildPrithviModel():
    backbone_args = dict(
        backbone_pretrained=True,
        backbone="prithvi_eo_v2_300_tl", # prithvi_eo_v2_300, prithvi_eo_v2_300_tl, prithvi_eo_v2_600, prithvi_eo_v2_600_tl
        backbone_coords_encoding=["time", "location"],
        backbone_bands=BANDS,
        backbone_num_frames=NUM_FRAMES,
    )

    decoder_args = dict(
        decoder="UperNetDecoder",
        decoder_channels=256,
        decoder_scale_modules=True,
    )

    necks = [
        dict(
                name="SelectIndices",
                # indices=[2, 5, 8, 11]    # indices for prithvi_vit_100
                indices=[5, 11, 17, 23],   # indices for prithvi_eo_v2_300
                # indices=[7, 15, 23, 31]  # indices for prithvi_eo_v2_600
            ),
        dict(
                name="ReshapeTokensToImage",
                effective_time_dim=NUM_FRAMES,
            )
        ]

    model_args = dict(
        **backbone_args,
        **decoder_args,
        num_classes=len(CLASS_WEIGHTS),
        head_dropout=HEAD_DROPOUT,
        necks=necks,
        rescale=True,
    )
    self.model = SemanticSegmentationTask(
        model_args=model_args,
        plot_on_val=False,
        class_weights=CLASS_WEIGHTS,
        loss="ce",
        lr=LR,
        optimizer="AdamW",
        optimizer_hparams=dict(weight_decay=WEIGHT_DECAY),
        ignore_index=-1,
        freeze_backbone=FREEZE_BACKBONE,
        freeze_decoder=False,
        model_factory="EncoderDecoderFactory"
    )

In [ ]:
# Logger
    logger = TensorBoardLogger(
        save_dir=OUT_DIR,
        name="multicrop_example",
    )

    # Trainer
    trainer = pl.Trainer(
        accelerator="auto",
        strategy="auto",
        devices="auto",
        precision="bf16-mixed",
        num_nodes=1,
        logger=logger,
        max_epochs=EPOCHS,
        check_val_every_n_epoch=1,
        log_every_n_steps=10,
        enable_checkpointing=False,
        limit_predict_batches=1,  # predict only in the first batch for generating plots
    )
trainerain.fit(model,datamodule=data_module)
Privpreds = trainer.predict(model, datamodule=data_module, ckpt_path=ckpt_path)

In [ ]:
dataModule.add(Privreds)

In [ ]:
def vgg_block(num_convs, out_channels):
    layers = []
    for _ in range(num_convs):
      layers.append(nn.LazyConv2d(out_channels, kernel_size=3, padding=1))
      layers.append(nn.ReLU())
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2))
    return nn.Sequential(*layers)

In [ ]:
class VGG(d2l.Classifier):


  def __init__(self, arch, num_classes=37):
        super().__init__()
        self.save_hyperparameters()
        conv_blks = []
        for (num_convs, out_channels) in arch:
            conv_blks.append(vgg_block(num_convs, out_channels))
        self.net = nn.Sequential(
            *conv_blks, nn.Flatten(),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(0.5),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(0.5),
            nn.LazyLinear(num_classes))
        self.net.apply(d2l.init_cnn)

In [ ]:
def train_model(model,epochs=30, lr=0.01, patience=5):
  model.to(device)
  optimizer = torch.optim.SGD(model.parameters(), lr=lr)
  loss_fn = nn.CrossEntropyLoss()
  # overall loss value for each epoch:
  best_accuracy = 0
  patience_counter = 0
  loss_train = []
  loss_valid = []
  accuracy_valid = []
  for epoch in range(epochs) :
    print("epoch", epoch + 1,)
    model.train()
    loss_values = [] # loss values for each batch
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)
        loss_values.append(loss.item())
        optimizer.zero_grad()
        with torch.no_grad():
            loss.backward()
            optimizer.step()
    loss_train.append(np.mean(loss_values))
    model.eval()
    loss_values = []
    accuracy_values = []
    for batch_X, batch_y in val_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        with torch.no_grad():
            preds = model(batch_X)
            loss = loss_fn(preds, batch_y)
            loss_values.append(loss.item())
            accuracy_values.append(model.accuracy(preds, batch_y).item())
    loss_valid.append(np.mean(loss_values))
    accuracy_valid.append(np.mean(accuracy_values))
    print ("accuracy: ", accuracy_valid[-1])
    if accuracy_valid[-1] < best_accuracy:
      patience_counter += 1
      if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch}.")
        break
    else:
      best_accuracy = accuracy_valid[-1]
      patience_counter = 0
  return loss_train, loss_valid, accuracy_valid

In [ ]:
train_loader = dataModule.get_dataloader(train=True)
val_loader = dataModule.get_dataloader(train=False)

model = CNNbaseline(lr=0.01)

loss_train, loss_valid, accuracy_valid = train_model(model, epochs=25, lr=0.01, patience=5)
torch.save(model.state_dict(), 'trained_models/baseline_model.pth')

###### https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-600M-TL


In [ ]:
API_KEY = "722001B7-D884-3902-A833-C05D71922F2B" 
COMMODITY = "CORN"
STAT = "PRODUCTION"
START_YEAR = 2025
END_YEAR = 2025
AGG_LEVEL = "STATE"
# Set up the API request
url = "https://quickstats.nass.usda.gov/api/api_GET/"
all_dfs = [] # Create an empty list for storing the results

# This program allows downloading multiple years data 
# Each year is downloaded separately

for year in range(START_YEAR, END_YEAR + 1):
    params = {
        "key": API_KEY,
        "commodity_desc": COMMODITY,
        "statisticcat_desc": STAT,
        "year": str(year),
        "agg_level_desc": AGG_LEVEL,
        "format": "JSON"
    }

    r = requests.get(url, params=params)
    data = r.json().get("data", [])
    temp = pd.DataFrame(data)

    # Make sure we got a response and append to the main list
    if not temp.empty:
        temp["year"] = year
        all_dfs.append(temp)

df = pd.concat(all_dfs, ignore_index=True) # Concatenate all dataframes
df.head() # Show the top rows
